# Data Exploration
Exploring Gmail data by importing authentication and fetching methods.

In [ ]:
from _typeshed import wsgi
# Import functions from the other files
from gmail_auth import authenticate_gmail
from fetch_gmail import fetch_recent_emails
from googleapiclient.discovery import build

def fetch_emails_under_each_label(max_results_per_label=2):
    """
    Authenticates and iterates through all Gmail labels, 
    fetching a small number of emails for each label to explore the data.
    """
    # 1. Authenticate using our existing function
    creds = authenticate_gmail()
    service = build("gmail", "v1", credentials=creds)
    
    # 2. Get all labels
    print("Fetching labels...")
    results = service.users().labels().list(userId="me").execute()
    labels = results.get("labels", [])
    
    if not labels:
        print("No labels found.")
        return
    
    # 3. Iterate over labels and fetch emails for each
    for label in labels:
        label_name = label['name']
        label_id = label['id']
        
        print(f"\n{'='*50}")
        print(f"LABEL: {label_name}")
        print(f"{'='*50}")
        
        try:
            # Query messages with this specific label ID
            msg_results = service.users().messages().list(
                userId='me', 
                labelIds=[label_id], 
                maxResults=max_results_per_label
            ).execute()
            
            messages = msg_results.get('messages', [])
            
            if not messages:
                print("  No messages found in this label.")
                continue
            
            for i, msg in enumerate(messages, 1):
                # Fetch message metadata
                msg_detail = service.users().messages().get(
                    userId="me", 
                    id=msg["id"], 
                    format="metadata", 
                    metadataHeaders=["Subject", "From", "Date"]
                ).execute()
                
                headers = msg_detail.get("payload", {}).get("headers", [])
                subject, sender, date = "(No Subject)", "(Unknown)", "(Unknown)"
                
                for header in headers:
                    name = header["name"].lower()
                    if name == "subject": subject = header["value"]
                    elif name == "from": sender = header["value"]
                    elif name == "date": date = header["value"]
                
                snippet = msg_detail.get("snippet", "")
                
                print(f"  [{i}] From: {sender}")
                print(f"      Date: {date}")
                print(f"      Subj: {subject}")
                print(f"      Body: {snippet[:80]}...")
                print("  -" * 25)
                
        except Exception as e:
            print(f"  Error fetching emails for {label_name}: {e}")

# Run the exploration
fetch_emails_under_each_label(max_results_per_label=2)


In [2]:
def count_emails_per_label():
    """
    Fetches and prints the total number of emails (messages) in each label.
    """
    creds = authenticate_gmail()
    service = build("gmail", "v1", credentials=creds)
    
    print("Fetching label statistics...\n")
    results = service.users().labels().list(userId="me").execute()
    labels = results.get("labels", [])
    
    if not labels:
        print("No labels found.")
        return
        
    print(f"{'-'*65}")
    print(f"{'Label Name':<30} | {'Total Messages':<15} | {'Unread Messages'}")
    print(f"{'-'*65}")
    
    for label in labels:
        label_id = label['id']
        label_name = label['name']
        
        try:
            # We need to get the specific label to see messagesTotal and messagesUnread
            # because list() doesn't always return these details.
            label_details = service.users().labels().get(userId="me", id=label_id).execute()
            
            total = label_details.get('messagesTotal', 0)
            unread = label_details.get('messagesUnread', 0)
            
            print(f"{label_name[:28]:<30} | {total:<15} | {unread}")
            
        except Exception as e:
            print(f"{label_name[:28]:<30} | Error fetching details: {e}")

count_emails_per_label()


Fetching label statistics...

-----------------------------------------------------------------
Label Name                     | Total Messages  | Unread Messages
-----------------------------------------------------------------
CHAT                           | 0               | 0
SENT                           | 464             | 0
INBOX                          | 12910           | 11169
IMPORTANT                      | 1399            | 726
TRASH                          | 0               | 0
DRAFT                          | 2               | 0
SPAM                           | 13              | 13
CATEGORY_FORUMS                | 0               | 0
CATEGORY_UPDATES               | 10719           | 9536
CATEGORY_PERSONAL              | 568             | 222
CATEGORY_PROMOTIONS            | 879             | 671
CATEGORY_SOCIAL                | 733             | 729
YELLOW_STAR                    | 0               | 0
STARRED                        | 35              | 0
UNREAD       